# LoRA Training — Methodology Reference (Colab T4)

> **Methodology reference. This app applies pre-trained LoRAs.**

This notebook is a **documented reference** for how a Stable Diffusion 1.5 LoRA is trained. It was **not executed by the author on custom data** — VoxelCraft applies pre-trained, license-verified LoRAs; it does not train them. The cells below are the standard approach, annotated, so the choices VoxelCraft makes when *applying* LoRAs (rank-agnostic loading, weights 0.0–1.5, stacking up to two) are grounded in how the artifacts they consume are produced.

Companion doc: [`docs/LORA_TRAINING.md`](../docs/LORA_TRAINING.md).

## 1. Environment (reference)

A free Colab **T4** (16 GB VRAM) is enough to train an SD 1.5 LoRA. The standard toolchain is the `diffusers` training example plus `peft` and `accelerate`.

In [ ]:
# Reference only — not executed by the author.
# !pip install -q diffusers[training] peft accelerate transformers bitsandbytes
# !accelerate config default

## 2. Dataset preparation

50–500 images at 512×512 (SD 1.5's native resolution), balanced across the concept, hand-curated to remove duplicates and artifacts. Each image gets a caption containing a rare **trigger word** plus descriptive attributes. A held-out 5–10% validation split detects overfitting during training rather than after.

In [ ]:
# Reference dataset layout (not executed):
#   data/train/img_0001.png   +   data/train/img_0001.txt  ("zxc painting, oil on canvas, ...")
#   data/train/img_0002.png   +   data/train/img_0002.txt
# Resolution 512, square (center-crop or pad); captions specific and trigger-word-inclusive.

## 3. Rank, alpha, and learning rate

- **Rank** is the dimensionality of the low-rank weight-update matrices — capacity vs file size. Typical: 4–8 (style/simple), 16–32 (the practical sweet spot), 64–128 (complex, overfitting-prone).
- **Alpha** scales the update at inference (`weight × alpha / rank`); `alpha ≈ rank` is empirically stable.
- **Learning rate** ~1e-4 to 5e-4 with AdamW (8-bit AdamW to save VRAM), a 10–20% warmup, and cosine/linear decay.

In [ ]:
# Reference training invocation (not executed) — diffusers SD 1.5 LoRA example:
# !accelerate launch train_text_to_image_lora.py \
#   --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
#   --train_data_dir="data/train" --resolution=512 --center_crop \
#   --rank=16 --learning_rate=1e-4 --lr_scheduler="cosine" --lr_warmup_steps=100 \
#   --train_batch_size=4 --max_train_steps=1000 --checkpointing_steps=200 \
#   --seed=42 --output_dir="lora-out"
# rank=16, alpha defaults to rank; ~1000 steps ≈ 2–5 epochs on a 200-image set.

## 4. Steps, overfitting, and checkpoints

For ~200 images, 500–1000 steps at batch size 4 is roughly 2–5 epochs. Overfitting shows as a widening train/validation loss gap or outputs that reproduce training images instead of generalizing. Checkpoint every 100–200 steps and pick the best snapshot before quality degrades.

## 5. How VoxelCraft applies the result

The output of the above is a small `.safetensors` adapter. VoxelCraft **applies** it — it does not produce it:

```python
pipe.load_lora_weights(repo_or_path, weight_name="adapter.safetensors", adapter_name="style")
pipe.enable_lora()
pipe.set_adapters(["style"], adapter_weights=[0.8])  # 0.0–1.5, up to two adapters
```

Every adapter in the registry is pre-trained by a third party and added only after its commercial-use license is verified by hand. The training methodology above is reference knowledge; the shipped, verifiable capability is application.